# LightGBM Model

In [1]:
# Reusable code lives in src/footy_prediction (install with `pip install -e .` from the repo root).
from footy_prediction.data_loading import load_features
from footy_prediction.evaluation import evaluate_holdout_model, evaluate_market_baseline
from footy_prediction.feature_sets import FEATURE_GROUPINGS
from footy_prediction.modeling import run_grouping_cross_validation

In [4]:
# load the features from data/features
features_df = load_features("features_data_xg_rolling")

In [12]:
features_df.columns.tolist() #2690
# (features_df['match_id']).max()
# for i in range(4):
#  print(i)
# a = [1,2,3,4]
# a.iloc[:2]

['match_id',
 'kickoff',
 'HomeTeam',
 'AwayTeam',
 'Div',
 'FTR',
 'Referee',
 'market_home_prob_fair',
 'market_draw_prob_fair',
 'market_away_prob_fair',
 'home_points_home_last_5',
 'away_points_away_last_5',
 'goals_difference_last_5',
 'points_difference_last_5',
 'home_fouls_for_last_5',
 'away_fouls_for_last_5',
 'home_corners_for_last_5',
 'away_corners_for_last_5',
 'rest_days_diff',
 'home_xg_last_5',
 'away_xg_last_5']

In [27]:
FEATURE_GROUPINGS['market_only']

['market_home_prob_fair', 'market_draw_prob_fair', 'market_away_prob_fair']

In [9]:
best_params_groupings = run_grouping_cross_validation(features_df, FEATURE_GROUPINGS)
best_params_groupings

Running cross-validation for grouping: market_only...


KeyError: "Missing required columns: ['season']"

In [15]:
# Evaluate the best grouping with its own CV-selected features and hyperparameters.
best = best_params_groupings.iloc[0]
evaluate_holdout_model(features_df, best["selected_features"], hyperparams=best["best_params"])

{'train_matches': 10121,
 'test_matches': 2531,
 'split_fraction': 0.8,
 'holdout_log_loss': 0.9995,
 'holdout_brier_score': np.float64(0.596),
 'model': LGBMClassifier(colsample_bytree=0.7, learning_rate=0.01, max_depth=4,
                min_child_samples=30, n_estimators=150, num_class=3,
                objective='multiclass', random_state=42, subsample=0.7,
                verbosity=-1)}

In [16]:
market_benchmark = evaluate_market_baseline(
    data=features_df,
    start_fraction=0.8,
    end_fraction=1.0,
    target_column="FTR",
    order_column="match_id"
)

print(market_benchmark)

{'match_count': 2531, 'start_percentile': '80%', 'end_percentile': '100%', 'market_log_loss': 0.9865, 'market_brier_score': np.float64(0.5878)}


# APPENDIX